In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv
from sklearn.metrics import r2_score
import numpy as np
import pickle
import random

# Set all RNGs before initializing each baseline model.
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Baseline graph convolution with pooled atom and optional molecular features.
class SimpleGCN(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h_dim in hidden_dims:
            self.convs.append(GCNConv(in_dim, h_dim))
            in_dim = h_dim
        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if hasattr(data, 'edge_attr') and data.edge_attr is not None and hasattr(self, 'edge_norm'):
            _ = self.edge_norm(data.edge_attr)
        u = data.u if hasattr(data, 'u') else None
        if u is not None and hasattr(self, 'global_norm'):
            u = self.global_norm(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if (u is not None and hasattr(self, 'global_mlp')) else node_pool
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out

def create_data_loader(graph_data, batch_size=32, shuffle=True):
    data_list = []
    for graph in graph_data:
        data_list.append(Data(
            x=graph['x'],
            edge_index=graph['edge_index'],
            edge_attr=graph.get('edge_attr', None),
            u=graph.get('u', None),
            y=graph['y']
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def train_model(
    train_data_dir: str,
    val_data_dir: str,
    save_path: str,
    epochs=1000,
    batch_size=32,
    lr=1e-4,
    min_lr=1e-6,
    hidden_dims=[64, 64],
    dropout=0.2,
    lr_patience=10,
    es_patience=100,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    train_path = os.path.join(train_data_dir, "graph_data.pt")
    val_path = os.path.join(val_data_dir, "graph_data.pt")
    for graph_path in (train_path, val_path):
        if not os.path.isfile(graph_path):
            raise FileNotFoundError(f"Required graph data not found: {graph_path}")
    # Graph files are PyTorch pickles: load only trusted repository-generated data.
    train_graph_data = torch.load(train_path, weights_only=False)
    val_graph_data = torch.load(val_path, weights_only=False)
    print(f"Loaded training data: {len(train_graph_data)} samples")
    print(f"Loaded validation data: {len(val_graph_data)} samples")

    # Fit the target transformation on training labels only.
    train_y = torch.stack([g['y'] for g in train_graph_data]).view(-1)
    y_mean, y_std = train_y.mean().item(), train_y.std().item() + 1e-8
    print(f"Data normalization - Mean: {y_mean:.4f}, Std: {y_std:.4f}")

    for g in train_graph_data:
        g['y'] = (g['y'] - y_mean) / y_std
    for g in val_graph_data:
        g['y'] = (g['y'] - y_mean) / y_std

    train_loader = create_data_loader(train_graph_data, batch_size, True)
    val_loader = create_data_loader(val_graph_data, batch_size, False)

    sample = train_graph_data[0]
    node_dim = sample['x'].size(1)
    edge_dim = sample.get('edge_attr').size(1) if (sample.get('edge_attr') is not None) else 0
    global_dim = sample.get('u').size(1) if (sample.get('u') is not None) else 0
    print(f"Feature dimensions - Node: {node_dim}, Edge: {edge_dim}, Global: {global_dim}")

    model = SimpleGCN(node_dim, edge_dim, global_dim, hidden_dims, dropout).to(device)
    print(f"Model initialized with hidden dims: {hidden_dims}, dropout: {dropout}")

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=lr_patience, min_lr=min_lr
    )

    best_r2, no_improve = -float('inf'), 0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0

        for batch in train_loader:
            batch = batch.to(device)
            pred, _ = model(batch, return_feat=True)
            loss = F.mse_loss(pred, batch.y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_epoch_loss = epoch_loss / len(train_loader)

        model.eval()
        ys_train, preds_train = [], []
        with torch.no_grad():
            for batch in train_loader:
                batch = batch.to(device)
                out = model(batch)
                ys_train.append(batch.y.view(-1).cpu().numpy())
                preds_train.append(out.cpu().numpy())
        train_r2 = r2_score(np.concatenate(ys_train), np.concatenate(preds_train))

        ys_val, preds_val = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                ys_val.append(batch.y.view(-1).cpu().numpy())
                preds_val.append(out.cpu().numpy())
        val_r2 = r2_score(np.concatenate(ys_val), np.concatenate(preds_val))

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:4d} | Loss: {avg_epoch_loss:.4f} | "
              f"Train R²: {train_r2:.4f} | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")

        scheduler.step(avg_epoch_loss)

        if val_r2 > best_r2:
            best_r2 = val_r2
            no_improve = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'y_mean': y_mean,
                'y_std': y_std,
                'node_dim': node_dim,
                'edge_dim': edge_dim,
                'global_dim': global_dim,
                'hidden_dims': hidden_dims,
                'dropout': dropout
            }, save_path)
            print(f"Best model saved! Val R² improved to {best_r2:.4f}")
        else:
            no_improve += 1
            if no_improve >= es_patience:
                print(f"Early stopping triggered: Val R² has no improvement for {es_patience} epochs.")
                break

    print(f"\nTraining complete | Best Val R²: {best_r2:.4f} | Model saved at: {save_path}")
    return best_r2, save_path

if __name__ == "__main__":
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    # All paths are relative to the project working directory.
    train_data_dir = os.path.join("data-set", "train")
    val_data_dir = os.path.join("data-set", "validation")
    save_dir = os.path.join("checkpoints", "baseline")
    os.makedirs(save_dir, exist_ok=True)

    for seed in seeds:
        print(f"\n===== Current Random Seed: {seed} =====")
        set_seed(seed)
        # The checkpoint name retains the baseline architecture and seed.
        out_path = os.path.join(save_dir, f"gcn_best_seed({seed})_128_128_0.1.pt")
        val_r2, _ = train_model(
            train_data_dir=train_data_dir,
            val_data_dir=val_data_dir,
            save_path=out_path,
            epochs=5000,
            batch_size=64,
            lr=1e-3,
            min_lr=1e-4,
            hidden_dims=[128, 128],
            dropout=0.1,
            lr_patience=20,
            es_patience=100
        )
        print(f"Seed {seed} → Validation R²: {val_r2:.4f}")
    print("Baseline training completed for all evaluation seeds.")
